# Model Missing `resultSectionalTime`

This notebook filters out race classes flagged in the notebook notes, keeps a compact set of predictive columns, applies light feature engineering, trains a Gradient Boosted Tree regressor, validates it on observed `resultSectionalTime` rows, and predicts `resultSectionalTime` for the rows where it is missing.

In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

In [2]:
target_col = "resultSectionalTime"
excluded_raceclass_patterns = ["OR", "D", "HD", "HP", "IT", "IV"]

selected_columns = [
    target_col,
    "resultPosition",
    "resultBtnDistance",
    "resultRunTime",
    "resultAdjustedTime",
    "resultDogWeight",
    "trapNumber",
    "raceDistance",
    "raceGoing",
    "raceWinTime",
    "numeratorSP",
    "denominatorSP",
    "commentScore",
    "raceType",
    "raceClass",
    "trackName",
    "resultComment",
    "SP",
    "raceDate",
]

dog_infos = pd.read_csv("../data/intermediate/03_dog_infos_processed.csv", usecols=selected_columns)

raceclass_exclusion_mask = dog_infos["raceClass"].isna()
for pattern in excluded_raceclass_patterns:
    raceclass_exclusion_mask |= dog_infos["raceClass"].str.contains(pattern, na=False)

dog_infos = dog_infos.loc[~raceclass_exclusion_mask].copy()
dog_infos.shape

(2584131, 19)

## Light Feature Engineering

Potential lightweight features for `resultSectionalTime`:

- Calendar features from `raceDate` such as month, weekday, and day-of-year.
- Odds-derived features from `numeratorSP` and `denominatorSP`, including a decimal-price proxy and implied probability.
- Relative timing features using `resultRunTime`, `resultAdjustedTime`, and `raceWinTime`.
- Simple decomposition of `raceClass` into a letter family and numeric grade.
- Small signal-extraction flags from `resultComment` instead of full text modeling.

In [3]:
model_df = dog_infos.copy()

model_df["raceDate"] = pd.to_datetime(model_df["raceDate"], errors="coerce")
model_df["race_year"] = model_df["raceDate"].dt.year
model_df["race_month"] = model_df["raceDate"].dt.month
model_df["race_dayofweek"] = model_df["raceDate"].dt.dayofweek
model_df["race_dayofyear"] = model_df["raceDate"].dt.dayofyear

model_df["sp_decimal"] = 1 + model_df["numeratorSP"].fillna(0) / model_df["denominatorSP"].replace(0, np.nan)
model_df["implied_prob"] = 1 / model_df["sp_decimal"]

model_df["run_time_minus_win_time"] = model_df["resultRunTime"] - model_df["raceWinTime"]
model_df["adjusted_time_minus_win_time"] = model_df["resultAdjustedTime"] - model_df["raceWinTime"]
model_df["run_speed"] = model_df["raceDistance"] / model_df["resultRunTime"].replace(0, np.nan)
model_df["adjusted_speed"] = model_df["raceDistance"] / model_df["resultAdjustedTime"].replace(0, np.nan)

model_df["raceClass_family"] = model_df["raceClass"].str.extract(r"([A-Za-z]+)", expand=False).fillna("Unknown")
model_df["raceClass_grade"] = pd.to_numeric(
    model_df["raceClass"].str.extract(r"(\d+)", expand=False),
    errors="coerce",
)

comment_text = model_df["resultComment"].fillna("").str.lower()
model_df["comment_len"] = comment_text.str.len()
model_df["comment_has_clear"] = comment_text.str.contains("clear|clrrun")
model_df["comment_has_crowded"] = comment_text.str.contains("crowd|crd")
model_df["comment_has_bump"] = comment_text.str.contains("bmp|baulk")
model_df["comment_has_early_pace"] = comment_text.str.contains("qaw|ep|led")
model_df["comment_has_wide"] = comment_text.str.contains("wide|mid")

feature_columns = [
    "resultRunTime",
    "resultAdjustedTime",
    "resultPosition",
    "resultBtnDistance",
    "resultDogWeight",
    "trapNumber",
    "raceDistance",
    "raceGoing",
    "raceWinTime",
    "numeratorSP",
    "denominatorSP",
    "commentScore",
    "raceType",
    "raceClass",
    "trackName",
    "resultComment",
    "SP",
    "race_year",
    "race_month",
    "race_dayofweek",
    "race_dayofyear",
    "sp_decimal",
    "implied_prob",
    "run_time_minus_win_time",
    "adjusted_time_minus_win_time",
    "run_speed",
    "adjusted_speed",
    "raceClass_family",
    "raceClass_grade",
    "comment_len",
    "comment_has_clear",
    "comment_has_crowded",
    "comment_has_bump",
    "comment_has_early_pace",
    "comment_has_wide",
]

model_df[feature_columns + [target_col]].head()

,resultRunTime,resultAdjustedTime,resultPosition,resultBtnDistance,resultDogWeight,trapNumber,raceDistance,raceGoing,raceWinTime,numeratorSP,...,adjusted_speed,raceClass_family,raceClass_grade,comment_len,comment_has_clear,comment_has_crowded,comment_has_bump,comment_has_early_pace,comment_has_wide,resultSectionalTime
0,28.730000,28.73000,4.0,2.0,34.1,5,480.0,0.0,28.24,5.0,...,16.707275,A,1.0,18,False,False,False,False,True,4.46
2,28.720000,28.72000,3.0,1.0,33.9,3,480.0,0.0,28.58,11.0,...,16.713092,A,1.0,15,False,True,False,False,True,4.53
3,28.855968,28.86369,6.0,10.0,34.1,4,480.0,0.0,28.51,1.0,...,16.629891,A,1.0,22,False,True,False,False,False,4.45
4,28.690000,28.49000,2.0,10.0,34.5,5,480.0,-20.0,28.63,11.0,...,16.848017,A,1.0,19,False,True,False,True,True,4.45
16,29.770000,29.77000,2.0,1.0,34.9,6,500.0,0.0,29.68,7.0,...,16.795432,A,1.0,13,True,False,False,False,True,3.62


In [4]:
observed_mask = model_df[target_col].notna()

train_val_df = model_df.loc[observed_mask, feature_columns + [target_col]].copy()
missing_target_df = model_df.loc[~observed_mask, feature_columns].copy()

X = train_val_df[feature_columns]
y = train_val_df[target_col]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

categorical_features = X.select_dtypes(include=["object", "bool"]).columns.tolist()
numeric_features = [col for col in feature_columns if col not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1,
                encoded_missing_value=-1,
            ),
            categorical_features,
        ),
        ("numeric", "passthrough", numeric_features),
    ],
    verbose_feature_names_out=False,
)

gbr = HistGradientBoostingRegressor(
    loss="squared_error",
    learning_rate=0.05,
    max_iter=300,
    max_leaf_nodes=63,
    min_samples_leaf=200,
    validation_fraction=None,
    random_state=42,
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", gbr),
    ]
)

model.fit(X_train, y_train)

val_predictions = model.predict(X_val)
validation_metrics = pd.Series(
    {
        "mae": mean_absolute_error(y_val, val_predictions),
        "rmse": np.sqrt(mean_squared_error(y_val, val_predictions)),
        "r2": r2_score(y_val, val_predictions),
        "train_rows": len(X_train),
        "val_rows": len(X_val),
        "prediction_rows": len(missing_target_df),
    },
    name="validation_metrics",
)

missing_target_predictions = pd.DataFrame(
    {
        "predicted_resultSectionalTime": model.predict(missing_target_df),
    },
    index=missing_target_df.index,
)

dog_infos_with_predictions = dog_infos.copy()
dog_infos_with_predictions["resultSectionalTime_gbt"] = dog_infos_with_predictions[target_col]
dog_infos_with_predictions.loc[
    missing_target_predictions.index,
    "resultSectionalTime_gbt",
] = missing_target_predictions["predicted_resultSectionalTime"]


/var/folders/m_/00q0qn914cs98zd_8p4zlxmc0000gn/T/ipykernel_23818/1416083976.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object", "bool"]).columns.tolist()


In [5]:
validation_metrics

mae                8.106173e-02
rmse               7.699437e-01
r2                 8.890602e-01
train_rows         2.032820e+06
val_rows           5.082060e+05
prediction_rows    4.310500e+04
Name: validation_metrics, dtype: float64

In [7]:
dog_infos_with_predictions.loc[missing_target_predictions.index, [target_col, "resultSectionalTime_gbt"]].head(10)

,resultSectionalTime,resultSectionalTime_gbt
137,NaN,3.937479
308,NaN,5.438920
322,NaN,2.374600
517,NaN,2.487269
552,NaN,3.968330
670,NaN,5.088511
717,NaN,3.328236
812,NaN,2.616022
839,NaN,2.520005
1045,NaN,4.160571


In [8]:
dog_infos_with_predictions.loc[missing_target_predictions.index].head(5)

,SP,resultPosition,resultBtnDistance,resultSectionalTime,resultComment,resultRunTime,resultDogWeight,resultAdjustedTime,trapNumber,raceDate,raceType,raceClass,raceDistance,raceGoing,raceWinTime,trackName,numeratorSP,denominatorSP,commentScore,resultSectionalTime_gbt
137,4/1,6.0,6.0,NaN,"Checked3,(HT)",29.010000,26.000000,29.01000,1,2022-07-09,Flat,A4,460.0,0.0,27.660000,Henlow,4.000000,1.000000,-1.0,3.937479
308,9/4F,3.0,0.2,NaN,ClrRun,28.850000,29.600000,28.95000,5,2022-08-07,Flat,A6,462.0,10.0,28.730000,Kinsley,9.000000,4.000000,-0.5,5.438920
322,NaN,0.0,0.0,NaN,NaN,28.855968,29.963729,28.86369,6,2020-12-30,Flat,B5,450.0,0.0,28.514841,Doncaster,6.626101,1.675337,0.0,2.374600
517,5/1,1.0,0.0,NaN,"Mid,ALed",28.970000,33.100000,28.97000,4,2014-09-26,Flat,A2,480.0,0.0,28.970000,Swindon,5.000000,1.000000,-0.5,2.487269
552,4/1,6.0,3.5,NaN,"LdT2,(HT)",29.250000,27.700000,29.25000,3,2019-04-23,Flat,A7,460.0,0.0,28.170000,Henlow,4.000000,1.000000,0.0,3.968330
